In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog.silver;

In [0]:
%sql
SELECT * FROM catalog.bronze.visits_raw LIMIT 10;

visit_id,patient_id,hospital_id,admission_date,discharge_date,admission_type,length_of_stay_days,diagnosis_code,ward_type,attending_physician_id,total_cost_php,payment_mode,insurance_claim_no,discharge_disposition,is_30day_readmission,_rescued_data,_source_file,_ingest_timestamp,ingestion_date
V00892,P0415,H001,03/16/2022,2022-03-19,Elective,3,K70,Private Room,DR005,23000,PhilHealth + OOP,CLM-5628326,Discharged - Improved,UNKNOWN,null,abfss://data@datastoragea.dfs.core.windows.net/staging/visits_raw/visits_raw,2026-05-05T10:58:19.890Z,2026-05-05
V01336,P0436,H003,2024-05-05,2024-05-08,Elective,3,I48,General Ward,DR020,21000,PhilHealth + HMO,CLM-7212969,Discharged - Improved,Y,null,abfss://data@datastoragea.dfs.core.windows.net/staging/visits_raw/visits_raw,2026-05-05T10:58:19.890Z,2026-05-05
V00499,P0172,H009,2024-07-21,2024-07-24,Emergency,3,Z51,Semi-Private,DR038,107000,PhilHealth Only,CLM-5960414,Discharged - Improved,N,null,abfss://data@datastoragea.dfs.core.windows.net/staging/visits_raw/visits_raw,2026-05-05T10:58:19.890Z,2026-05-05
V01123,P0099,H003,April 03 2023,2023-04-07,Emergency,4,I10,Private Room,DR003,37000,Out-of-Pocket,null,Discharged - Improved,UNKNOWN,null,abfss://data@datastoragea.dfs.core.windows.net/staging/visits_raw/visits_raw,2026-05-05T10:58:19.890Z,2026-05-05
V00972,P0272,H009,08/27/2024,2024-08-30,Emergency,3,O14,General Ward,DR017,24000,PhilHealth + HMO,CLM-7405836,Discharged - Improved,UNKNOWN,null,abfss://data@datastoragea.dfs.core.windows.net/staging/visits_raw/visits_raw,2026-05-05T10:58:19.890Z,2026-05-05
V00357,P0267,H009,2023-12-27,2023-12-29,Elective,2,E11,General Ward,DR034,35000,PhilHealth Only,CLM-2786778,Discharged - Stable,N,null,abfss://data@datastoragea.dfs.core.windows.net/staging/visits_raw/visits_raw,2026-05-05T10:58:19.890Z,2026-05-05
V00346,P0227,H004,03/19/2024,2024-03-31,Emergency,12,I63,General Ward,DR035,96000,PhilHealth + HMO,CLM-3935644,Discharged - Stable,UNKNOWN,null,abfss://data@datastoragea.dfs.core.windows.net/staging/visits_raw/visits_raw,2026-05-05T10:58:19.890Z,2026-05-05
V01930,P0418,H005,28-06-2022,2022-06-30,Elective,2,I10,Semi-Private,DR049,49000,PhilHealth + HMO,CLM-5901411,Discharged - Improved,UNKNOWN,null,abfss://data@datastoragea.dfs.core.windows.net/staging/visits_raw/visits_raw,2026-05-05T10:58:19.890Z,2026-05-05
V00113,P0382,H002,2024-07-30,2024-08-03,Elective,4,I48,General Ward,DR004,31000,PhilHealth Only,CLM-1959545,Discharged - Improved,N,null,abfss://data@datastoragea.dfs.core.windows.net/staging/visits_raw/visits_raw,2026-05-05T10:58:19.890Z,2026-05-05
V01041,P0430,H007,2022-05-12,2022-05-15,Emergency,3,J18,General Ward,DR032,31000,Out-of-Pocket,null,Discharged - Improved,N,null,abfss://data@datastoragea.dfs.core.windows.net/staging/visits_raw/visits_raw,2026-05-05T10:58:19.890Z,2026-05-05


In [0]:
from pyspark.sql.functions import col, lag, try_to_date, coalesce, datediff, when, current_timestamp, to_date
from delta.tables import DeltaTable
from pyspark.sql.window import Window

In [0]:
bronze_table = 'catalog.bronze.visits_raw'
silver_table = 'catalog.silver.fact_visits'
checkpoint_path = "abfss://data@datastoragea.dfs.core.windows.net/silver/fact_visits/checkpoint/"

In [0]:
silver_patients_table = "catalog.silver.dim_patients"
silver_diagnosis_table = "catalog.silver.dim_diagnosis"
silver_hospitals_table = "catalog.silver.dim_hospitals"
silver_physicians_table = "catalog.silver.dim_physicians"

In [0]:
# Load columns needed from each silver table

# For patient data
df_patients = (spark.read.table("catalog.silver.dim_patients")
    .select("patient_id", "first_name", "last_name", "gender", "dob", "city", "assigned_branch", "patient_name_hash")
    .withColumnRenamed("city", "patient_city")
    .withColumnRenamed("assigned_branch", "patient_assigned_branch"))

# For diagnosis-level readmission KPIs
df_diagnosis = (spark.read.table("catalog.silver.dim_diagnosis")
    .select("diagnosis_code", "diagnosis_desc", "diagnosis_category", "readmission_risk_level"))

# For hospital and hospital region KPIs
df_hospitals = (spark.read.table("catalog.silver.dim_hospitals")
    .select("hospital_id", "hospital_name", "city", "network_region_cluster", "hospital_class", "bed_count")
    .withColumnRenamed("city", "hospital_city"))

# For physician KPI
df_physicians = (spark.read.table("catalog.silver.dim_physicians")
    .select("physician_id", "specialization")
    .withColumnRenamed("physician_id", "attending_physician_id"))

In [0]:
df_visits_bronze = (
    spark.readStream.table(bronze_table)
)

In [0]:
window = Window.partitionBy("patient_id").orderBy("admission_date")

# Clean visits stream
df_visits_clean = (
    df_visits_bronze
        .drop(
            "_rescued_data",
            "_source_file",
            "_ingest_timestamp",
            "ingestion_date",
            "days_since_last_discharge", # recomputed from clean dates
            "discharge_disposition",     
            "ward_type",
            "admission_type",
            "payment_mode",
            "insurance_claim_no"
        )
        .withColumn("admission_date", coalesce(
            try_to_date(col("admission_date"), "yyyy-MM-dd"),
            try_to_date(col("admission_date"), "MM/dd/yyyy"),
            try_to_date(col("admission_date"), "dd-MM-yyyy"),
            try_to_date(col("admission_date"), "MMMM dd yyyy")))
        .withColumn("discharge_date", coalesce(
            try_to_date(col("discharge_date"), "yyyy-MM-dd"),
            try_to_date(col("discharge_date"), "MM/dd/yyyy"),
            try_to_date(col("discharge_date"), "dd-MM-yyyy"),
            try_to_date(col("discharge_date"), "MMMM dd yyyy")))
        # Recompute LOS from clean dates
        .withColumn("length_of_stay_days",
            when(
            col("discharge_date") >= col("admission_date"),
            datediff(col("discharge_date"), col("admission_date"))
        ).otherwise(None))
        .withColumn("is_30day_readmission",
            when(col("is_30day_readmission") == "Y", True)
            .when(col("is_30day_readmission") == "N", False)
            .otherwise(None))
        )

# Join clean fact visit with dimension tables
df_fact_combined = (
    df_visits_clean
        .join(df_patients, "patient_id", "left")
        .join(df_hospitals, "hospital_id", "left")
        .join(df_diagnosis, "diagnosis_code", "left")
        .join(df_physicians, "attending_physician_id", "left")
        .withColumn("admission_date", to_date("admission_date"))
        .withColumn("discharge_date", to_date("discharge_date"))
        .withColumn("load_timestamp", current_timestamp())
        )

In [0]:
# Merge into silver fact visit
def merge_fact_visits(batch_df, batch_id):
    # Deduplicate on visit_id — keep latest ingested
    batch_df = batch_df.dropDuplicates(["visit_id"])

    # Recompute is_30day_readmission from clean dates
    window = Window.partitionBy("patient_id").orderBy("admission_date")

    batch_df = (batch_df
        .withColumn("prev_discharge_date",
            lag("discharge_date", 1).over(window))
        .withColumn("days_since_last_discharge",
            datediff(col("admission_date"), col("prev_discharge_date")))
        .withColumn("is_30day_readmission",
            when(col("days_since_last_discharge").between(0, 30), True)
            .otherwise(False))
        .drop("prev_discharge_date")    # intermediate calculation column - help compute days_since_last_discharge and is_30day_readmission
        .withColumn("load_timestamp", current_timestamp()))
    

    if not spark.catalog.tableExists(silver_table):
        # First run — create the table
        (batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table))  
        return
    
    # Load Delta table by name and upsert into Silver
    fact = DeltaTable.forName(spark, silver_table)

    (fact.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.visit_id = s.visit_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute())

In [0]:
# Run as availableNow increment
(
    df_fact_combined.writeStream
        .foreachBatch(merge_fact_visits)  # for every batch, merge_fact_visits is run
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)

In [0]:
%sql
SELECT * FROM catalog.silver.fact_visits LIMIT 10;

attending_physician_id,diagnosis_code,hospital_id,patient_id,visit_id,admission_date,discharge_date,length_of_stay_days,total_cost_php,is_30day_readmission,first_name,last_name,gender,dob,patient_city,patient_assigned_branch,patient_name_hash,hospital_name,hospital_city,network_region_cluster,hospital_class,bed_count,diagnosis_desc,diagnosis_category,readmission_risk_level,specialization,load_timestamp,days_since_last_discharge
null,C34,H001,P0179,V00236,2022-12-22,2022-12-28,6,149000,false,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon General Hospital - Manila Flagship,Manila,NCR Cluster,Tertiary,350,Lung Cancer,Oncology,High,null,2026-05-24T14:24:56.656Z,null
DR031,A41,H007,P0179,V00376,2023-01-11,2023-01-24,13,101000,true,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon Community Hospital - Iloilo,Iloilo City,Visayas Cluster,Secondary,130,Sepsis,Critical Care,High,Cardiology,2026-05-24T14:24:56.656Z,14
DR046,J18,H002,P0179,V00558,2023-02-14,2023-02-22,8,16000,true,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon General Hospital - Quezon City,Quezon City,NCR Cluster,Tertiary,280,Pneumonia,Respiratory,Medium,Nephrology,2026-05-24T14:24:56.656Z,21
DR049,A15,H001,P0179,V02123,2023-05-22,2023-05-24,2,50000,false,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon General Hospital - Manila Flagship,Manila,NCR Cluster,Tertiary,350,Pulmonary Tuberculosis,Respiratory,Medium,Oncology,2026-05-24T14:24:56.656Z,89
DR033,I50,H001,P0179,V00619,2023-06-07,2023-06-12,5,54000,true,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon General Hospital - Manila Flagship,Manila,NCR Cluster,Tertiary,350,Heart Failure,Cardiovascular,High,Obstetrics,2026-05-24T14:24:56.656Z,14
DR037,C50,H009,P0179,V00929,2023-07-09,2023-07-14,5,40000,true,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon Community Hospital - Cagayan de Oro,Cagayan de Oro,Mindanao Cluster,Secondary,110,Breast Cancer,Oncology,High,Infectious Disease,2026-05-24T14:24:56.656Z,27
DR048,Z49,H005,P0179,V02549,2023-07-12,2023-07-14,2,17000,false,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon Community Hospital - Pampanga,"San Fernando, Pampanga",Luzon Cluster,Secondary,150,Dialysis Care Encounter,Procedure,High,Infectious Disease,2026-05-24T14:24:56.656Z,-2
DR008,J18,H005,P0179,V02100,2024-12-12,2024-12-18,6,24000,false,Christian,Aquino,M,1972-04-14,Iligan,H007,47a2de2a99379f79c9a9a5b766c7d7ea5b75e535d9337baae70266858e5c1efc,MediLuzon Community Hospital - Pampanga,"San Fernando, Pampanga",Luzon Cluster,Secondary,150,Pneumonia,Respiratory,Medium,Gastroenterology,2026-05-24T14:24:56.656Z,517
DR020,A15,H005,P0243,V01294,2023-03-28,2023-03-30,2,38000,false,Veronica,Morales,F,1964-08-22,Cebu City,H004,6615e4dc33b6be458341cd540677c11626de0f519a9d910e01d75ec1d29ea908,MediLuzon Community Hospital - Pampanga,"San Fernando, Pampanga",Luzon Cluster,Secondary,150,Pulmonary Tuberculosis,Respiratory,Medium,Neurology,2026-05-24T14:24:56.656Z,null
DR046,Z51,H003,P0243,V01822,2023-04-08,2023-04-09,1,76000,true,Veronica,Morales,F,1964-08-22,Cebu City,H004,6615e4dc33b6be458341cd540677c11626de0f519a9d910e01d75ec1d29ea908,MediLuzon Specialist Center - Makati,Makati,NCR Cluster,Specialty,180,Chemotherapy / Radiotherapy,Procedure,High,Nephrology,2026-05-24T14:24:56.656Z,9


In [0]:
# Data quality check
from pyspark.sql.functions import col, count, sum as spark_sum, min, max, avg, round

print("FACT VISITS")
df = spark.read.table("catalog.silver.fact_visits")
total = df.count()

print(f"Total rows: {total}")
print("-" * 50)

# Null check
print("\nNull counts:")
for c in ["visit_id", "patient_id", "hospital_id", "diagnosis_code",
          "admission_date", "discharge_date", "is_30day_readmission",
          "length_of_stay_days", "total_cost_php"]:
    n = df.filter(col(c).isNull()).count()
    print(f"  {'OK' if n == 0 else 'CHECK'} {c}: {n}")

# Duplicates
dupes = total - df.select("visit_id").distinct().count()
print(f"\nDuplicate visit_ids: {'OK' if dupes == 0 else dupes}")

# Readmission split
print("\nReadmission breakdown:")
df.groupBy("is_30day_readmission").count().show()

# LOS and cost ranges
print("LOS range:")
df.select(
    min("length_of_stay_days").alias("min"),
    max("length_of_stay_days").alias("max"),
    round(avg("length_of_stay_days"), 1).alias("avg")
).show()

print("Cost range (PHP):")
df.select(
    min("total_cost_php").alias("min"),
    max("total_cost_php").alias("max"),
    round(avg("total_cost_php"), 0).alias("avg")
).show()

# Readmission rate by risk level
print("Readmission rate by risk level:")
(df.groupBy("readmission_risk_level")
   .agg(
       count("visit_id").alias("visits"),
       spark_sum(col("is_30day_readmission").cast("int")).alias("readmissions")
   )
   .withColumn("rate_%", round(col("readmissions") / col("visits") * 100, 1))
   .orderBy("rate_%", ascending=False)
   .show())

FACT VISITS
Total rows: 3000
--------------------------------------------------

Null counts:
  OK visit_id: 0
  OK patient_id: 0
  OK hospital_id: 0
  OK diagnosis_code: 0
  OK admission_date: 0
  OK discharge_date: 0
  OK is_30day_readmission: 0
  CHECK length_of_stay_days: 29
  OK total_cost_php: 0

Duplicate visit_ids: OK

Readmission breakdown:
+--------------------+-----+
|is_30day_readmission|count|
+--------------------+-----+
|               false| 1853|
|                true| 1147|
+--------------------+-----+

LOS range:
+---+---+---+
|min|max|avg|
+---+---+---+
|  0| 14|5.2|
+---+---+---+

Cost range (PHP):
+-----+-----+-------+
|  min|  max|    avg|
+-----+-----+-------+
|10000|99000|57548.0|
+-----+-----+-------+

Readmission rate by risk level:
+----------------------+------+------------+------+
|readmission_risk_level|visits|readmissions|rate_%|
+----------------------+------+------------+------+
|                  High|  1492|         689|  46.2|
|                   Lo